# CubbyLLM 0068: route 5 on an A100 (grilly2)

The home card's three timings, run once on Colab's A100 with grilly2 built here from source:
1. **the head alone** (`head_lab_time.py --route card`): route 5 against a full 32k head and the full 128k head;
2. **inside a real decode step** (`head_lab_decode.py`): at 128k and on the synthetic 256k head;
3. **through the wired `generate`** (`head_lab_generate.py`): the seats' own call.

About 25 minutes: Rust and the Vulkan SDK (2 min), grilly2's build (5 to 10 min), the export (2 min), the three runs
(about 10 min). Nothing is trained and no weight changes. The logs land in `MyDrive/cubbyllm/head_shortlist/a100/`;
Cowork reads them at home and commits them. **Use an A100 runtime, run all, then disconnect the runtime.**

grilly2 has not run on Colab before (grilly v1 has); cell 3 stops with a clear message if Vulkan does not see the card.

In [ ]:
# --- 1. the code: CubbyLLM and grilly2 at the commits the order names ---
import os, sys, glob, json, shutil, hashlib, zipfile, datetime, subprocess
from google.colab import drive; drive.mount('/content/drive')
CUBBY_BRANCH = 'cowork/0068'
G2_BRANCH, G2_COMMIT = 'cowork/shortlist-generate', '7cb24f8'

def run(args, cwd=None, env=None):
    print('$', ' '.join(args), flush=True)
    p = subprocess.Popen(args, cwd=cwd, env=env, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='')
    assert p.wait() == 0, f'exit {p.returncode}: {" ".join(args[:3])}'

if not os.path.exists('/content/CubbyLLM'):
    run(['git', 'clone', '-q', '-b', CUBBY_BRANCH, '--single-branch', 'https://github.com/Grillcheese-AI/CubbyLLM.git',
         '/content/CubbyLLM'])
else:
    run(['git', 'pull', '-q', '--ff-only'], cwd='/content/CubbyLLM')
if not os.path.exists('/content/grilly2'):
    run(['git', 'clone', '-q', '--recursive', '-b', G2_BRANCH, 'https://github.com/Grillcheese-AI/grilly2.git',
         '/content/grilly2'])
run(['git', 'checkout', '-q', G2_COMMIT], cwd='/content/grilly2')
run(['git', 'log', '--oneline', '-1'], cwd='/content/CubbyLLM')
run(['git', 'log', '--oneline', '-1'], cwd='/content/grilly2')
run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'])

In [ ]:
# --- 2. build grilly2: Rust, the LunarG SDK (headers, glslc, shaderc), maturin ---
SDK_VER = '1.4.335.0'                                   # grilly2's CI pins this one
ROOT = '/content/vulkansdk'
if not glob.glob(f'{ROOT}/*/x86_64/bin/glslc'):
    os.makedirs(ROOT, exist_ok=True)
    tar = f'vulkansdk-linux-x86_64-{SDK_VER}.tar.xz'
    run(['curl', '-fsSL', '-o', f'/content/{tar}', f'https://sdk.lunarg.com/sdk/download/{SDK_VER}/linux/{tar}'])
    run(['tar', '-xJf', f'/content/{tar}', '-C', ROOT])
SDK = f'{ROOT}/{SDK_VER}/x86_64'
if not os.path.exists(f'{SDK}/lib/libshaderc_combined.a'):   # a source-only tarball: build what grilly2 needs
    run(['bash', './vulkansdk', '-j', str(os.cpu_count()), 'vulkan-headers', 'vulkan-loader', 'shaderc'],
        cwd=f'{ROOT}/{SDK_VER}')
assert os.path.exists(f'{SDK}/bin/glslc') and os.path.exists(f'{SDK}/lib/libshaderc_combined.a'), \
    f'the SDK is not where expected: look in {ROOT}'
CARGO = os.path.expanduser('~/.cargo/bin')
if not os.path.exists(f'{CARGO}/cargo'):
    run(['bash', '-c', 'curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'maturin'])
BUILD = dict(os.environ, VULKAN_SDK=SDK, PATH=f'{CARGO}:{SDK}/bin:' + os.environ['PATH'])
shutil.rmtree('/content/g2dist', ignore_errors=True)
run([sys.executable, '-m', 'maturin', 'build', '--release', '--out', '/content/g2dist', '--auditwheel', 'skip'],
    cwd='/content/grilly2', env=BUILD)
wheel = glob.glob('/content/g2dist/*.whl')[0]
with zipfile.ZipFile(wheel) as z:                       # the extension beside the checkout's python/, as at home,
    for m in z.namelist():                              # so each log names grilly2's commit
        if m.startswith('grilly/_C'):
            open(f'/content/grilly2/python/{m}', 'wb').write(z.read(m))
print('built', os.path.basename(wheel))

In [ ]:
# --- 3. Vulkan sees the A100? (the loader from apt, NVIDIA's ICD; grilly2 refuses a CPU driver) ---
run(['bash', '-c', 'apt-get -qq update && apt-get -qq install -y libvulkan1 vulkan-tools > /dev/null'])
NV = [d for d in ('/usr/lib64-nvidia', '/usr/lib/x86_64-linux-gnu') if glob.glob(f'{d}/libGLX_nvidia.so*')]
icds = glob.glob('/usr/share/vulkan/icd.d/nvidia*.json') + glob.glob('/etc/vulkan/icd.d/nvidia*.json')
if not icds and NV:                                     # Colab mounts the driver without its ICD file
    os.makedirs('/etc/vulkan/icd.d', exist_ok=True)
    json.dump({'file_format_version': '1.0.0', 'ICD': {'library_path': 'libGLX_nvidia.so.0', 'api_version': '1.3'}},
              open('/etc/vulkan/icd.d/nvidia_icd.json', 'w'))
    print('wrote /etc/vulkan/icd.d/nvidia_icd.json')
ENV = dict(os.environ, PYTHONPATH='/content/grilly2/python', PYTHONIOENCODING='utf-8',
           LD_LIBRARY_PATH=':'.join(NV + [os.environ.get('LD_LIBRARY_PATH', '')]))
subprocess.run(['vulkaninfo', '--summary'], env=ENV)
doc = subprocess.run([sys.executable, '-m', 'grilly.doctor'], env=ENV, text=True, capture_output=True)
print(doc.stdout, doc.stderr)
lines = doc.stdout.splitlines()
device = next((x for x in lines if x.startswith('[ok]') and ' device ' in x), '')
ran = any(x.startswith('[ok]') and 'a kernel ran' in x for x in lines)
assert 'NVIDIA' in device and ran, (
    'grilly2 does not see the A100 through Vulkan. Send Cowork the output above (vulkaninfo and grilly.doctor).')

In [ ]:
# --- 4. the inputs: the base exported from the checkpoint 0068 read, the route's index, the 50 timed states ---
DRIVE = '/content/drive/MyDrive/cubbyllm'
CK450 = '/content/base450m_cont_final.pt'
if not os.path.exists(CK450):                           # Drive-FUSE reads are slow: one copy to the local disk
    shutil.copy(f'{DRIVE}/runs/base450m_cont/base450m_cont_final.pt', CK450)
EXPORT = '/content/export'
if not os.path.exists(f'{EXPORT}/model.safetensors'):
    run([sys.executable, 'validation/export_base.py', '--ckpt', CK450, '--out', EXPORT], cwd='/content/CubbyLLM')
from safetensors import safe_open
import numpy as np
with safe_open(f'{EXPORT}/model.safetensors', framework='numpy') as f:
    head = np.ascontiguousarray(f.get_tensor('lm_head.weight'), dtype=np.float32)
HEAD_SHA = '63fcdfe23e7e16617443d2c4b26db743bb89146d56bbb7d03f3c755584580cd1'   # the home export's head rows
assert head.shape == (128000, 1024) and hashlib.sha256(head.tobytes()).hexdigest() == HEAD_SHA, \
    "the exported head is not the one the route's index was built on"
del head
DUMPS = '/content/head_shortlist'
os.makedirs(DUMPS, exist_ok=True)
for name in ('index_router_succ.npz', 'heldout_b.npz'):
    if not os.path.exists(f'{DUMPS}/{name}'):
        shutil.copy(f'{DRIVE}/head_shortlist/{name}', f'{DUMPS}/{name}')
SHA = {'index_router_succ.npz': '814c8e03245161591d697724a1931e01549e22e52d06012ed4b99d6913fbdd15',
       'heldout_b.npz': 'eef2d754ba806991707ec4deff9edeeced277b72a51e3bcfbc42ca3df76de448'}
for name, want in SHA.items():
    assert hashlib.sha256(open(f'{DUMPS}/{name}', 'rb').read()).hexdigest() == want, f'{name} is not the one recorded'
CACHE = f'{DRIVE}/token_cache_cont'
print('inputs checked')

In [ ]:
# --- 5. the three timings (about 10 min), as at home ---
common = ['--export', EXPORT, '--dumps', DUMPS]
run([sys.executable, '-u', 'validation/head_lab_time.py', '--route', 'card', '--dump', 'heldout_b'] + common,
    cwd='/content/CubbyLLM', env=ENV)
run([sys.executable, '-u', 'validation/head_lab_decode.py', '--cache', CACHE] + common, cwd='/content/CubbyLLM', env=ENV)
run([sys.executable, '-u', 'validation/head_lab_generate.py', '--cache', CACHE] + common, cwd='/content/CubbyLLM',
    env=ENV)

In [ ]:
# --- 6. keep the logs on Drive (named _a100), with the card and driver they ran on ---
OUT = f'{DRIVE}/head_shortlist/a100'
os.makedirs(OUT, exist_ok=True)
today = datetime.date.today().isoformat()
logs = [p for k in ('head_lab_time_grilly_card', 'head_lab_decode', 'head_lab_generate')
        for p in glob.glob(f'/content/CubbyLLM/validation/logs/{k}_{today}_{G2_COMMIT}.json')]
assert len(logs) == 3, logs
for p in logs:
    shutil.copy(p, f'{OUT}/' + os.path.basename(p).replace('.json', '_a100.json'))
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], text=True,
                     capture_output=True).stdout
open(f'{OUT}/a100_env_{today}.txt', 'w').write(smi + '\n' + doc.stdout)
print('kept on Drive:', sorted(os.listdir(OUT)))
print('Done: disconnect the runtime.')